# YOLO / RT-DETR experiment

Configuration selects the model, loss and training budget. Shared COCO evaluation uses the selected final checkpoint on original fold-0 images.

## Environment

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

KAGGLE = Path('/kaggle/working').is_dir()
WORK = Path('/kaggle/working') if KAGGLE else Path.cwd()
REPO = WORK / 'ai-hw-object-detection' if KAGGLE else (WORK if (WORK / 'src').is_dir() else WORK.parent)
if not KAGGLE:
    WORK = REPO
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Stalex575/ai-hw-object-detection.git', str(REPO)], check=True)
if KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)
sys.path.insert(0, str(REPO))
for name in ('YOLO_CONFIG_DIR', 'MPLCONFIGDIR'):
    directory = WORK / 'outputs' / name.lower()
    directory.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(directory)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())


## Data and run configuration

In [ ]:
import json
import yaml
from src import evaluate, submit, yolo

roots = [REPO / 'data', REPO]
if KAGGLE:
    roots += [p.parent for p in Path('/kaggle/input').rglob('test_images.csv')]
DATA = next(p for p in roots if (p / 'train/annotations_coco.json').is_file())
PREP = WORK / 'prep' if KAGGLE else REPO / 'prep'
RUN_NAME = os.environ.get('RUN_NAME', 'yolo26s_smoke_fold0_640_s42_v1')
CONFIG = Path(os.environ.get('RUN_CONFIG', 'configs/yolo26_smoke.yaml'))
if not CONFIG.is_absolute():
    CONFIG = REPO / CONFIG
config = yaml.safe_load(CONFIG.read_text())
OUT = WORK / 'outputs' / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
FOLD, FRAME_STEP = config.get('fold', 0), config.get('frame_step', 5)
COMPARE_NMS = False
import logging
logger = logging.getLogger('hw2')
logger.setLevel(logging.INFO)
logger.propagate = False
for previous in list(logger.handlers):
    logger.removeHandler(previous)
    previous.close()
file_handler = logging.FileHandler(OUT / 'run.log')
file_handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
logger.addHandler(file_handler)
logger.info('Data=%s; fold=%s; frame_step=%s', DATA, FOLD, FRAME_STEP)
print('Data:', DATA, '\nRun:', OUT)


## Fold preparation and evaluator check

In [ ]:
DATA_YAML, GT_PATH, fold_info = yolo.prepare_fold(DATA, PREP, REPO / 'folds.csv', FOLD, FRAME_STEP)
gt = evaluate.read_json(GT_PATH)
oracle = evaluate.evaluate_predictions(gt, evaluate.oracle_predictions(gt))
assert oracle['metrics']['mAP'] > 0.999
print(fold_info)
print('GT sanity check:', oracle['metrics'])
evaluate.write_json(OUT / 'preflight.json', {**fold_info, 'oracle': oracle['metrics']})

logger.info('Preflight: %s; oracle: %s', fold_info, oracle['metrics'])


## Training

In [ ]:
import csv
from contextlib import redirect_stdout, redirect_stderr
import gc
import hashlib
from importlib.metadata import version
import platform
import time
import torch
from ultralytics import YOLO, RTDETR, settings
from ultralytics.utils import LOGGER
from src.yolo_nwd import configure as configure_nwd, configure_mps_assignment

settings.update({'wandb': False})
recipe = dict(config)
model_class = recipe.pop('model_class', 'yolo')
Model = {'yolo': YOLO, 'rtdetr': RTDETR}[model_class]
eval_weights = recipe.pop('eval_weights', 'last')
nwd_alpha, nwd_constant = recipe.pop('nwd_alpha', None), recipe.pop('nwd_constant', 12.8)
configure_nwd(nwd_alpha, nwd_constant)
for key in ('fold', 'frame_step'):
    recipe.pop(key, None)
weights, hours = recipe.pop('model'), recipe.pop('wall_time_hours', None)
DEVICE = recipe.get('device', 0)
mps_assign_per_image = recipe.pop('mps_assign_per_image', False)
configure_mps_assignment(str(DEVICE) == 'mps' and mps_assign_per_image)
HALF = str(DEVICE) not in ('mps', 'cpu')
if not HALF:
    recipe.update(amp=False, workers=0)
INFERENCE_BATCH = min(recipe.get('batch', 4), 8)
recipe.update(data=str(DATA_YAML), project=str(OUT), name='train', exist_ok=False)
resume = os.environ.get('RESUME_TRAINING') == '1'
eval_only = os.environ.get('EVAL_ONLY') == '1'
checkpoint = OUT / 'train/weights/last.pt'
if not eval_only:
    assert resume or not (OUT / 'train').exists(), 'Run output already exists'
    model = Model(checkpoint if resume else weights)
    gpu = torch.cuda.get_device_name(DEVICE) if HALF else str(DEVICE)
    runtime = {'python': platform.python_version(), 'gpu': gpu,
               'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip(),
               'versions': {name: version(name) for name in ('torch', 'torchvision', 'numpy', 'ultralytics', 'faster-coco-eval')},
               'config': config, 'resume': resume,
               'source_sha256': {str(path.relative_to(REPO)): hashlib.sha256(path.read_bytes()).hexdigest()
                                 for path in [REPO / 'src/yolo.py', REPO / 'src/yolo_nwd.py', REPO / 'notebooks/04_anchor_free_yolo.ipynb']},
               'weights_sha256': hashlib.sha256(Path(model.ckpt_path).read_bytes()).hexdigest(),
               'folds_sha256': hashlib.sha256((REPO / 'folds.csv').read_bytes()).hexdigest()}
    evaluate.write_json(OUT / 'runtime.json', runtime)
    deadline = [None]
    batches = [0]
    timed_out = [False]

    def start_budget(trainer):
        deadline[0] = time.perf_counter() + hours * 3600 if hours else None

    def progress(trainer):
        timed_out[0] = deadline[0] is not None and time.perf_counter() >= deadline[0]
        trainer.stop |= timed_out[0]
        stop_file = os.environ.get('TRAIN_STOP_FILE')
        trainer.stop |= bool(stop_file and Path(stop_file).exists())
        batches[0] += 1
        if batches[0] % 50 == 0 or trainer.stop:
            evaluate.write_json(OUT / 'progress.json', {'epoch': trainer.epoch + 1, 'batches': batches[0],
                                'time': time.time(), 'stage': 'training', 'timed_out': timed_out[0]})

    def checkpoint_saved(trainer):
        evaluate.write_json(OUT / 'progress.json', {'epoch': trainer.epoch + 1, 'time': time.time(), 'stage': 'checkpoint'})
        periodic = sorted((OUT / 'train/weights').glob('epoch*.pt'), key=lambda path: path.stat().st_mtime)
        for old in periodic[:-2]:
            old.unlink()

    model.add_callback('on_train_start', start_budget)
    model.add_callback('on_train_batch_end', progress)
    model.add_callback('on_model_save', checkpoint_saved)
    started = time.perf_counter()
    console_handlers = LOGGER.handlers[:]
    LOGGER.handlers = [file_handler]
    try:
        with redirect_stdout(file_handler.stream), redirect_stderr(file_handler.stream):
            if resume:
                model.train(resume=True, device=DEVICE, batch=recipe['batch'])
            else:
                model.train(**recipe)
    except BaseException:
        logger.exception('Training failed')
        raise
    finally:
        LOGGER.handlers = console_handlers
    training_hours = (time.perf_counter() - started) / 3600
    with model.trainer.csv.open() as stream:
        completed_epochs = int(list(csv.DictReader(stream))[-1]['epoch'])
    evaluate.write_json(OUT / 'training.json', {'hours': training_hours, 'epochs': completed_epochs,
                                              'target_epochs': recipe['epochs'], 'actual_batch': model.trainer.batch_size,
                                              'timed_out': timed_out[0]})
    del model
    gc.collect()
    if str(DEVICE) == 'mps':
        torch.mps.empty_cache()
else:
    runtime = evaluate.read_json(OUT / 'runtime.json')
    training_hours = evaluate.read_json(OUT / 'training.json')['hours']
BEST = OUT / 'train/weights' / f'{eval_weights}.pt'
assert BEST.is_file(), 'Evaluation checkpoint missing'
evaluate.write_json(OUT / 'progress.json', {'time': time.time(), 'stage': 'export'})
print('Training complete; evaluating', BEST.name)


## Shared validation and inference heads

In [ ]:
predictions = yolo.predict(Model(BEST), gt['images'], DATA / 'train', OUT / 'val_predictions.json', imgsz=recipe['imgsz'], device=DEVICE, half=HALF, batch=INFERENCE_BATCH)
metrics = evaluate.evaluate_predictions(gt, predictions)
evaluate.write_json(OUT / 'metrics.json', metrics)
print(metrics['metric_status'])
print(metrics['metrics'])
if COMPARE_NMS:
    nms_predictions = yolo.predict(Model(BEST), gt['images'], DATA / 'train', OUT / 'val_predictions_nms.json',
                                   imgsz=recipe['imgsz'], device=DEVICE, half=HALF, batch=INFERENCE_BATCH, end2end=False)
    nms_metrics = evaluate.evaluate_predictions(gt, nms_predictions)
    evaluate.write_json(OUT / 'metrics_nms.json', nms_metrics)
logger.info('Shared validation metrics: %s', metrics['metrics'])

gc.collect()
if str(DEVICE) == "mps":
    torch.mps.empty_cache()


## Validation examples and batch-one speed

In [ ]:
from PIL import Image, ImageDraw
from src.viz_gt import grid

examples, clips = [], set()
for image in gt['images']:
    clip = Path(image['file_name']).parent.name
    if clip in clips:
        continue
    clips.add(clip)
    picture = Image.open(DATA / 'train' / image['file_name']).convert('RGB')
    draw = ImageDraw.Draw(picture)
    targets = [a for a in gt['annotations'] if a['image_id'] == image['id'] and not a.get('iscrowd')]
    detected = [p for p in predictions if p['image_id'] == image['id'] and p['score'] >= 0.1]
    for records, color in ((targets, 'lime'), (detected, 'red')):
        for record in records:
            x, y, w, h = record['bbox']
            draw.rectangle((x, y, x + w, y + h), outline=color, width=3)
    draw.text((10, 10), clip + ' | green: GT, red: predictions', fill='white', stroke_width=2, stroke_fill='black')
    examples.append(picture)
    if len(examples) == 4:
        break
grid(examples, cols=2, tile_w=640).save(OUT / 'validation_examples.jpg', quality=90)
paths = [DATA / 'train' / image['file_name'] for image in gt['images']]
sample = [paths[round(i * (len(paths) - 1) / 31)] for i in range(32)]
speed = yolo.benchmark(Model(BEST), sample, imgsz=recipe['imgsz'], device=DEVICE, half=HALF)
evaluate.write_json(OUT / 'speed.json', speed)
logger.info('Speed: %s', speed)
print(speed)

gc.collect()
if str(DEVICE) == "mps":
    torch.mps.empty_cache()


## Test predictions and submission

In [ ]:
test_images = submit.read_test_images(DATA / 'test_images.csv')
test_predictions = yolo.predict(Model(BEST), test_images, DATA, OUT / 'test_predictions.json', imgsz=recipe['imgsz'], device=DEVICE, half=HALF, batch=INFERENCE_BATCH)
submit.write_submission(test_images, test_predictions, OUT / 'submission.csv')
submission_check = submit.validate_submission(test_images, OUT / 'submission.csv')
assert submission_check['detections'] > 0
evaluate.write_json(OUT / 'submission_check.json', submission_check)
print(submission_check)
logger.info('Submission check: %s', submission_check)

gc.collect()
if str(DEVICE) == "mps":
    torch.mps.empty_cache()


## Run artifacts

In [ ]:
import csv
import zipfile
import shutil

row = {'id': RUN_NAME, 'owner': 'Oleh', 'family': model_class, 'fold': FOLD,
       **metrics['metrics'], 'FPS': speed['fps'], 'GPU_h_training': training_hours,
       'public_LB': '', 'commit': runtime['commit'], 'eval_weights': eval_weights,
       **evaluate.read_json(OUT / 'training.json')}
with (OUT / 'experiment.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(row))
    writer.writeheader()
    writer.writerow(row)
submission_copy = OUT.parent / f'{RUN_NAME}_submission.csv'
shutil.copy2(OUT / 'submission.csv', submission_copy)
archive = OUT.parent / f'{RUN_NAME}_results.zip'
logger.info('Run complete; results archive=%s', archive)
file_handler.flush()
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in OUT.rglob('*'):
        if path.is_file() and not (path.name.startswith('epoch') and path.suffix == '.pt'):
            bundle.write(path, path.relative_to(OUT))
evaluate.write_json(OUT / 'progress.json', {'time': time.time(), 'stage': 'done'})
print('Submission:', submission_copy)
print('Run archive:', archive)
